# Mode-Shift Opportunity — Category Deep Dive
Replicates the *Deep Dive for Florian* quad-chart analysis across four
commodity groups: **Copper (74)**, **Iron & Steel (72/73)**,
**Electronic Machinery (85)**, and **Apparel (61)**.

Each group maps every HS-4 code to one of four functional categories
(`raw`, `intermediate`, `consumer product`, `waste`) and then produces:
1. Quad chart — mode-shift opportunity, CO₂ savings, air-weight share, $/kg
2. Adjusted quad chart — 15 pp penalty for intermediate/raw goods (delay sensitivity)
3. Summary table

In [2]:
%load_ext autoreload
%autoreload 2

import os, sys, numpy as np
import pandas as pd
import plotly.graph_objects as go
import kaleido
from plotly.subplots import make_subplots
from pyproj import Geod
from tqdm import tqdm
from pathlib import Path

# Derive repo root from this notebook's location.
# Notebooks live in Scripts/, so repo root is one level up.
# Works regardless of where Jupyter was launched from.
NOTEBOOK_DIR = Path(os.path.abspath(''))
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'Scripts' else NOTEBOOK_DIR
SCRIPT_DIR = REPO_ROOT / 'Scripts'

# Add Scripts/ to the Python path so functions.py can be imported directly,
# then set the working directory to repo root (required by functions.py for
# all relative file paths like 'fuel_energy_info/', 'Maps/', etc.)
sys.path.insert(0, str(SCRIPT_DIR))
os.chdir(REPO_ROOT)

import functions

# Path to the 4-digit HTS dataset (not included in repo — too large for GitHub).
# Place all_4digits_data.xlsx in the repo root, or see thesisplots.ipynb for
# instructions on rebuilding it from the Census API.
DATA_PATH = REPO_ROOT / 'all_4digits_data.xlsx'

print('Repo root:', REPO_ROOT)
print('Working directory:', os.getcwd())

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Working directory: /Users/aidangoldenberg-hart/Documents/GitHub/Maritime_Aviation_Repo


## Helper Functions

In [3]:
geod = Geod(ellps='WGS84')

def great_circle_points(lon1, lat1, lon2, lat2, n_points=60):
    points = geod.npts(lon1, lat1, lon2, lat2, n_points)
    lons = [lon1] + [p[0] for p in points] + [lon2]
    lats = [lat1] + [p[1] for p in points] + [lat2]
    fixed_lons, fixed_lats = [lons[0]], [lats[0]]
    for i in range(1, len(lons)):
        if abs(lons[i] - lons[i-1]) > 180:
            fixed_lons.append(None)
            fixed_lats.append(None)
        fixed_lons.append(lons[i])
        fixed_lats.append(lats[i])
    return fixed_lons, fixed_lats

In [35]:
def map_top_flows_hts4(df, hts4_code, n=10):
    """Plot the top-n carbon-saving trade flows for a single HTS-4 code."""
    data = df.copy()
    data['carbon_savings'] = data['MtCO2eq_aviation'] - data['hypo_MtCO2eq_maritime']
    sub = data[data['HTS4'] == hts4_code].dropna(
        subset=['PORT_LAT','PORT_LON','dest_lat','dest_lon']
    )
    flows = sub.groupby(
        ['FLOW_TYPE','PORT_NAME','PORT_LAT','PORT_LON','CTY_NAME','dest_lat','dest_lon'],
        as_index=False
    ).agg({'carbon_savings': 'sum'})
    top = flows.sort_values('carbon_savings', ascending=False).head(n)
    top['line_width'] = 2 + 10 * (top['carbon_savings'] / top['carbon_savings'].max())

    fig = go.Figure()
    for _, row in top.iterrows():
        if row['FLOW_TYPE'] == 'exports':
            olat, olon, dlat, dlon = row['PORT_LAT'], row['PORT_LON'], row['dest_lat'], row['dest_lon']
            oname, dname = row['PORT_NAME'], row['CTY_NAME']
        else:
            olat, olon, dlat, dlon = row['dest_lat'], row['dest_lon'], row['PORT_LAT'], row['PORT_LON']
            oname, dname = row['CTY_NAME'], row['PORT_NAME']
        label = f'{oname} → {dname}'
        lons, lats = great_circle_points(olon, olat, dlon, dlat)
        fig.add_trace(go.Scattermapbox(
            lon=lons, lat=lats, mode='lines',
            line=dict(width=row['line_width']), opacity=0.7, name=label,
            hovertemplate=f'{label}<br>Savings: {row["carbon_savings"]:.3f} MtCO₂e<extra></extra>'
        ))
        fig.add_trace(go.Scattermapbox(
            lon=[dlon], lat=[dlat], mode='markers',
            marker=dict(size=12, symbol='triangle', color='red'), showlegend=False
        ))
    fig.update_layout(
        mapbox_style='carto-positron', mapbox_zoom=1, mapbox_center={'lat':20,'lon':0},
        title=f'Top {n} Carbon-Saving Flows — HTS4 {hts4_code}',
        legend_title='Origin → Destination', margin={'r':0,'t':40,'l':0,'b':0}
    )
    fig.show()

In [36]:
def analyze_hs2_group(hs2_list, all_hts_datasets, group_name='Commodity Group', n_sims=5000):
    """
    Combine HS-2 chapters, run Monte Carlo transport-cost comparison, and return
    a results dict with raw data, grouped summaries, MC results, and mode-shift
    opportunity tables.
    """
    # --- combine ---
    df = pd.concat(
        [all_hts_datasets[f'all_{code}_data'] for code in hs2_list],
        ignore_index=True
    )
    df = functions.general_emissions(df.copy())

    # --- aggregate to HS4 ---
    agg_cols = ['AIR_VAL_YR','AIR_WGT_YR','VES_VAL_YR','VES_WGT_YR',
                'MtCO2eq_maritime','MtCO2eq_aviation',
                'tkm_aviation','tkm_container_eff','tkm_noncontainer_eff']
    grouped = df.groupby('HTS4')[agg_cols].sum().reset_index()
    grouped['air_share'] = grouped['AIR_WGT_YR'] / (
        grouped['VES_WGT_YR'] + grouped['AIR_WGT_YR']
    )
    grouped = grouped.sort_values('air_share', ascending=False)

    # --- filter routes with air ---
    df_nonzeronan = df[
        df['haversine_distance_km'].notna() & (df['AIR_VAL_YR'] != 0)
    ]

    # --- Monte Carlo ---
    mc_results = functions.monte_carlo_transport_cost(df_nonzeronan, n_sims=n_sims, scc=0.0)
    median_val = mc_results['mode_shift_opp_share'].median()

    # --- median simulation ---
    median_sim_idx = (
        mc_results.assign(diff=lambda x: (x['mode_shift_opp_share'] - median_val).abs())
        .sort_values('diff').iloc[0]['sim']
    )
    median_params = mc_results.loc[mc_results['sim'] == median_sim_idx].iloc[0]

    hts_error_map = df_nonzeronan[['hts_num','error_i']].drop_duplicates().set_index('hts_num')['error_i']
    np.random.seed(int(median_sim_idx))
    error_draws_median = np.random.choice(hts_error_map.values, size=len(df_nonzeronan), replace=True)

    # --- apply median params ---
    air_cheaper_median = functions.total_transport_cost(
        df_nonzeronan,
        av_cost=median_params['av_cost'], mar_cost=median_params['mar_cost'],
        mar_speed=median_params['mar_speed'], av_speed=800,
        error_draws=error_draws_median, scc=0.0,
    )
    shifts = functions.hypothetical_mode_shift_emissions(air_cheaper_median)
    shifts_grouped = shifts.groupby('HTS4', as_index=False).agg(
        hypo_MtCO2eq_maritime=('hypo_MtCO2eq_maritime','sum'),
        MtCO2eq_aviation_shiftable=('MtCO2eq_aviation','sum')
    )

    acm_grouped = air_cheaper_median.groupby('HTS4').agg(
        **{c: (c,'sum') for c in agg_cols},
        mode_shift_opportunity=('mode_shift_opportunity','mean')
    ).reset_index()
    acm_grouped = acm_grouped.merge(shifts_grouped, on='HTS4', how='left')
    acm_grouped['carbon_savings'] = (
        acm_grouped['MtCO2eq_aviation_shiftable'] - acm_grouped['hypo_MtCO2eq_maritime']
    )
    acm_grouped = acm_grouped.sort_values('mode_shift_opportunity', ascending=False)

    # --- all routes with median params ---
    hts_error_map_all = df[['hts_num','error_i']].drop_duplicates().set_index('hts_num')['error_i']
    np.random.seed(int(median_sim_idx))
    error_draws_all = np.random.choice(hts_error_map_all.values, size=len(df), replace=True)
    mode_shift_allroutes = functions.total_transport_cost(
        df, av_cost=median_params['av_cost'], mar_cost=median_params['mar_cost'],
        mar_speed=median_params['mar_speed'], av_speed=800,
        error_draws=error_draws_all, scc=0.0,
    )

    return {
        'raw_data': df, 'grouped': grouped,
        'mc_results': mc_results, 'median_params': median_params,
        'air_cheaper_median': air_cheaper_median,
        'mode_shift_grouped': acm_grouped,
        'mode_shift_allroutes': mode_shift_allroutes,
        'shifts': shifts,
    }

In [37]:
import re

def _safe_filename(group_name, title_suffix):
    raw = f"{group_name}{title_suffix}"
    # replace any character that isn't alphanumeric, space, hyphen, or underscore
    return re.sub(r"[^\w\s-]", "", raw).strip().replace(" ", "_")

In [43]:
COLOR_MAP = {
    'raw':             '#8c564b',
    'intermediate':    '#1f77b4',
    'consumer product':'#2ca02c',
    'waste':           '#7f7f7f',
}

CO2_YRANGE = [0, 0.5]   # fixed range for all CO₂ savings panels

def deep_dive_quad_chart(results, category_map, group_name,
                         intermediate_threshold=0.15):
    # ── merge category labels ──
    df = results['mode_shift_grouped'].copy()
    df['HTS4'] = df['HTS4'].astype(str)

    cat_df = pd.DataFrame(list(category_map.items()), columns=['HTS4','category'])
    cat_df['HTS4'] = cat_df['HTS4'].astype(str)
    df = df.merge(cat_df, on='HTS4', how='left')
    df['category'] = df['category'].fillna('intermediate')

    # ── merge air_share from raw grouped ──
    raw_grouped = results['grouped'].copy()
    raw_grouped['HTS4'] = raw_grouped['HTS4'].astype(str)
    raw_grouped['air_share'] = (
        raw_grouped['AIR_WGT_YR'] /
        (raw_grouped['AIR_WGT_YR'] + raw_grouped['VES_WGT_YR'])
    )
    df = df.merge(raw_grouped[['HTS4','air_share']], on='HTS4', how='left')

    # ── derived metrics ──
    df['value_per_kg'] = df['AIR_VAL_YR'] / df['AIR_WGT_YR']

    # ── threshold adjustment ──
    df['modified_mode_shift_opp'] = df['mode_shift_opportunity']
    mask = df['category'].isin(['intermediate','raw'])
    df.loc[mask, 'modified_mode_shift_opp'] = (
        df.loc[mask, 'mode_shift_opportunity'] - intermediate_threshold
    ).clip(lower=0)

    df['adjusted_carbon_savings'] = 0.0
    consumer_mask     = df['category'].isin(['consumer product','waste'])
    intermediate_mask = df['category'].isin(['intermediate','raw'])
    df.loc[consumer_mask, 'adjusted_carbon_savings'] = df.loc[consumer_mask, 'carbon_savings']
    valid = intermediate_mask & (df['mode_shift_opportunity'] > 0)
    df.loc[valid, 'adjusted_carbon_savings'] = (
        df.loc[valid, 'carbon_savings']
        * df.loc[valid, 'modified_mode_shift_opp']
        / df.loc[valid, 'mode_shift_opportunity']
    )

    # ── single combined chart ──
    _six_panel_chart(df, group_name, intermediate_threshold)

    # ── summary table ──
    summary = (
        df.groupby('category')
        .agg(
            n_codes              =('HTS4','count'),
            total_air_wgt        =('AIR_WGT_YR','sum'),
            avg_air_share        =('air_share','mean'),
            avg_value_per_kg     =('value_per_kg','mean'),
            avg_mode_shift_opp   =('mode_shift_opportunity','mean'),
            total_carbon_savings =('carbon_savings','sum'),
            total_adj_savings    =('adjusted_carbon_savings','sum'),
        )
        .reset_index()
    )
    print(f'\n=== Summary — {group_name} ===')
    print(summary.to_string(index=False))
    print(f'  Raw carbon savings:      {df["carbon_savings"].sum():.4f} MtCO₂e')
    print(f'  Adjusted carbon savings: {df["adjusted_carbon_savings"].sum():.4f} MtCO₂e')

    return df


def _six_panel_chart(df, group_name, intermediate_threshold=0.15):
    """2-column × 3-row panel chart combining raw and adjusted views."""
    pct = int(intermediate_threshold * 100)
    co2_max = max(df['carbon_savings'].max(), df['adjusted_carbon_savings'].max())
    co2_yrange = [0, co2_max * 1.1]   # 10% headroom above the tallest bar

    fig = make_subplots(
        rows=3, cols=2,
        shared_xaxes=True,
        vertical_spacing=0.09,
        horizontal_spacing=0.08,
        subplot_titles=(
            'Share of Weight Flown by Air',
            'Value per kg of Air Freight ($/kg)',
            'Mode-Shift Opportunity',
            'Potential CO\u2082 Savings from Mode Shift (MtCO\u2082e)',
            f'Mode-Shift Opportunity  [\u2212{pct} pp for intermediate/raw]',
            f'Potential CO\u2082 Savings — Adjusted  [\u2212{pct} pp for intermediate/raw]',
        )
    )

    for cat in df['category'].dropna().unique():
            sub   = df[df['category'] == cat]
            color = COLOR_MAP.get(cat, 'gray')
            kw    = dict(marker_color=color, legendgroup=cat, name=cat)

            # Row 1 — showlegend=True here gives each category one legend entry
            fig.add_trace(go.Bar(x=sub['HTS4'].astype(str), y=sub['air_share'],
                                showlegend=True, **kw), row=1, col=1)
            fig.add_trace(go.Bar(x=sub['HTS4'].astype(str), y=sub['value_per_kg'],
                                showlegend=False, **kw), row=1, col=2)

            fig.add_trace(go.Bar(x=sub['HTS4'].astype(str), y=sub['mode_shift_opportunity'],
                                showlegend=False, **kw), row=2, col=1)
            fig.add_trace(go.Bar(x=sub['HTS4'].astype(str), y=sub['carbon_savings'],
                                showlegend=False, **kw), row=2, col=2)

            fig.add_trace(go.Bar(x=sub['HTS4'].astype(str), y=sub['modified_mode_shift_opp'],
                                showlegend=False, **kw), row=3, col=1)
            fig.add_trace(go.Bar(x=sub['HTS4'].astype(str), y=sub['adjusted_carbon_savings'],
                                showlegend=False, **kw), row=3, col=2)

    fig.update_layout(
        title=f'Mode Shift & Economic Characteristics — {group_name}',
        template='plotly_white',
        height=1200,
        legend=dict(title='Category', orientation='v', x=1.02, xanchor='left'),
        margin=dict(b=120, t=80, l=80, r=160),
    )

    # y-axes
    fig.update_yaxes(title_text='Air Weight Share',  range=[0, 1],        row=1, col=1)
    fig.update_yaxes(title_text='$/kg',                                    row=1, col=2)
    fig.update_yaxes(title_text='Frac. Could Shift', range=[0, 1],        row=2, col=1)
    fig.update_yaxes(title_text='MtCO\u2082e Saved', range=co2_yrange,   row=2, col=2)
    fig.update_yaxes(title_text='Frac. Could Shift', range=[0, 1],        row=3, col=1)
    fig.update_yaxes(title_text='MtCO\u2082e Saved', range=co2_yrange,   row=3, col=2)

    # x-axes — only label bottom row
    for c in [1, 2]:
        fig.update_xaxes(title_text='HTS4 Code', tickangle=45, row=3, col=c)

    fig.show()

    n_codes = df['HTS4'].nunique()
    fig.write_image(
        f"Thesis_Plots/{_safe_filename(group_name, '')}.png",
        width=max(1600, n_codes * 45),
        height=1400,
        scale=2,
    )

## Data Loading

In [7]:
xls = pd.ExcelFile(DATA_PATH)
all_hts_datasets = {
    sheet: pd.read_excel(xls, sheet_name=sheet)
    for sheet in xls.sheet_names
}
print('Sheets loaded:', list(all_hts_datasets.keys()))

Sheets loaded: ['all_01_data', 'all_02_data', 'all_03_data', 'all_04_data', 'all_05_data', 'all_06_data', 'all_07_data', 'all_08_data', 'all_09_data', 'all_10_data', 'all_11_data', 'all_12_data', 'all_13_data', 'all_14_data', 'all_15_data', 'all_16_data', 'all_17_data', 'all_18_data', 'all_19_data', 'all_20_data', 'all_21_data', 'all_22_data', 'all_23_data', 'all_24_data', 'all_25_data', 'all_26_data', 'all_27_data', 'all_28_data', 'all_29_data', 'all_30_data', 'all_31_data', 'all_32_data', 'all_33_data', 'all_34_data', 'all_35_data', 'all_36_data', 'all_37_data', 'all_38_data', 'all_39_data', 'all_40_data', 'all_41_data', 'all_42_data', 'all_43_data', 'all_44_data', 'all_45_data', 'all_46_data', 'all_47_data', 'all_48_data', 'all_49_data', 'all_50_data', 'all_51_data', 'all_52_data', 'all_53_data', 'all_54_data', 'all_55_data', 'all_56_data', 'all_57_data', 'all_58_data', 'all_59_data', 'all_60_data', 'all_61_data', 'all_62_data', 'all_63_data', 'all_64_data', 'all_65_data', 'all_66_d

---
## 1. Copper (HS Chapter 74)
| Category | Codes |
|---|---|
| raw | 7401 (mattes), 7402 (unrefined), 7403 (refined unwrought) |
| waste | 7404 (scrap & waste) |
| intermediate | 7405–7416, 7419 (alloys, wire, sheet, tube, fittings…) |
| consumer product | 7417 (cooking apparatus), 7418 (kitchen/household articles) |

In [8]:
copper_results = analyze_hs2_group(
    hs2_list=[74],
    all_hts_datasets=all_hts_datasets,
    group_name='Copper'
)

  Loading distance cache from ocean_distances_cache.csv...
    Found 18944 cached routes
  Total unique routes: 2660
  Already cached: 2660
  Need to calculate: 0


/Users/aidangoldenberg-hart/Documents/GitHub/Maritime_Aviation_Repo/Scripts/functions.py:1165: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df['tau_binned'] = df.groupby(bins)[tau_col].transform('median')


Total hypothetical maritime emissions under mode shift: 0.00089558 MtCO2eq
Total aviation emissions for these shipments: 0.1150 MtCO2eq
Differential emissions (aviation - hypothetical maritime): 0.1141 MtCO2eq


In [44]:
copper_category_map = {
    # raw materials
    '7401': 'raw',          # copper mattes; cement copper
    '7402': 'raw',          # unrefined copper; copper anodes for refining
    '7403': 'raw',          # refined copper, unwrought
    # waste
    '7404': 'waste',        # copper waste and scrap
    # intermediate goods
    '7405': 'intermediate', # master alloys of copper
    '7406': 'intermediate', # copper powders and flakes
    '7407': 'intermediate', # copper bars, rods, profiles
    '7408': 'intermediate', # copper wire
    '7409': 'intermediate', # copper plates, sheets, strip, foil (>0.15 mm)
    '7410': 'intermediate', # copper foil (<=0.15 mm)
    '7411': 'intermediate', # copper tubes and pipes
    '7412': 'intermediate', # copper tube or pipe fittings
    '7413': 'intermediate', # stranded wire, cables, plaited bands of copper
    '7415': 'intermediate', # nails, tacks, drawing pins, screws of copper
    '7418': 'consumer product', # table, kitchen, household articles of copper
    '7419': 'intermediate', # other articles of copper
}

copper_deep_dive = deep_dive_quad_chart(
    copper_results,
    copper_category_map,
    group_name='Copper (HS 74)'
)


=== Summary — Copper (HS 74) ===
        category  n_codes  total_air_wgt  avg_air_share  avg_value_per_kg  avg_mode_shift_opp  total_carbon_savings  total_adj_savings
consumer product        1         302977       0.040058         58.514283            0.222571              0.001101           0.001101
    intermediate       11       23761074       0.070255         29.652228            0.413506              0.104197           0.062746
             raw        3        1290396       0.058087          8.700842            0.633918              0.007201           0.005469
           waste        1         311187       0.000364         28.156498            0.625000              0.001593           0.001593
  Raw carbon savings:      0.1141 MtCO₂e
  Adjusted carbon savings: 0.0709 MtCO₂e


---
## 2. Iron & Iron/Steel Articles (HS Chapters 72 & 73)
| Category | Codes (selected) |
|---|---|
| raw | 7201–7203, 7206, 7218, 7224 (pig iron, ferro-alloys, ingots, billets) |
| waste | 7204 (ferrous waste & scrap) |
| intermediate | 7205, 7207–7217, 7219–7223, 7225–7229, 7301–7318, 7320, 7325–7326 |
| consumer product | 7319 (needles/hooks), 7321 (stoves), 7322 (radiators), 7323 (kitchen ware), 7324 (sanitary ware) |

In [ ]:
# ironsteel_results = analyze_hs2_group(
#     hs2_list=[72, 73],
#     all_hts_datasets=all_hts_datasets,
#     group_name='Iron and Steel'
# )

  Loading distance cache from ocean_distances_cache.csv...
    Found 18944 cached routes
  Total unique routes: 6679
  Already cached: 6679
  Need to calculate: 0


/Users/aidangoldenberg-hart/Documents/GitHub/Maritime_Aviation_Repo/Scripts/functions.py:1165: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



KeyboardInterrupt: 

In [ ]:
# ironsteel_category_map = {
#     # ── HS 72: Iron and Steel ──────────────────────────────────────────────
#     '7201': 'raw',          # pig iron and spiegeleisen
#     '7202': 'raw',          # ferro-alloys
#     '7203': 'raw',          # ferrous products by direct reduction of iron ore
#     '7204': 'waste',        # ferrous waste and scrap
#     '7205': 'intermediate', # granules and powders of pig iron
#     '7206': 'raw',          # iron/non-alloy steel in ingots (not further worked)
#     '7207': 'intermediate', # semi-finished products of iron/non-alloy steel
#     '7208': 'intermediate', # flat-rolled, hot-rolled, >=600mm, not clad
#     '7209': 'intermediate', # flat-rolled, cold-rolled, >=600mm
#     '7210': 'intermediate', # flat-rolled, >=600mm, plated or coated
#     '7211': 'intermediate', # flat-rolled, <600mm, not clad
#     '7212': 'intermediate', # flat-rolled, <600mm, clad or plated
#     '7213': 'intermediate', # bars and rods in coils, hot-rolled
#     '7214': 'intermediate', # other bars and rods of non-alloy steel
#     '7215': 'intermediate', # other bars and rods of other alloy steel
#     '7216': 'intermediate', # angles, shapes, sections
#     '7217': 'intermediate', # wire of non-alloy steel
#     '7218': 'raw',          # stainless steel in ingots or other primary forms
#     '7219': 'intermediate', # flat-rolled stainless, >=600mm
#     '7220': 'intermediate', # flat-rolled stainless, <600mm
#     '7221': 'intermediate', # bars/rods of stainless in coils
#     '7222': 'intermediate', # other bars/rods of stainless
#     '7223': 'intermediate', # wire of stainless steel
#     '7224': 'raw',          # other alloy steel in ingots
#     '7225': 'intermediate', # flat-rolled other alloy steel, >=600mm
#     '7226': 'intermediate', # flat-rolled other alloy steel, <600mm
#     '7227': 'intermediate', # bars/rods of other alloy steel in coils
#     '7228': 'intermediate', # other bars/rods of other alloy steel
#     '7229': 'intermediate', # wire of other alloy steel
#     # ── HS 73: Articles of Iron or Steel ──────────────────────────────────
#     '7301': 'intermediate', # sheet piling; welded angles
#     '7302': 'intermediate', # railway track construction material
#     '7303': 'intermediate', # tubes/pipes of cast iron
#     '7304': 'intermediate', # tubes/pipes, seamless
#     '7305': 'intermediate', # tubes/pipes, large diameter welded
#     '7306': 'intermediate', # other tubes/pipes/hollow profiles
#     '7307': 'intermediate', # tube/pipe fittings
#     '7308': 'intermediate', # structures (bridges, lock gates, towers…)
#     '7309': 'intermediate', # reservoirs, tanks, vats (capacity >300 L)
#     '7310': 'intermediate', # cans, boxes, cases, drums (capacity <=300 L)
#     '7311': 'intermediate', # containers for compressed or liquefied gas
#     '7312': 'intermediate', # stranded wire, ropes, cables of iron/steel
#     '7313': 'intermediate', # barbed wire; twisted fencing wire
#     '7314': 'intermediate', # cloth, grill, netting of iron wire
#     '7315': 'intermediate', # chain and parts
#     '7316': 'intermediate', # anchors, grapnels
#     '7317': 'intermediate', # nails, tacks, drawing pins (industrial use)
#     '7318': 'intermediate', # screws, bolts, nuts, washers
#     '7319': 'consumer product', # sewing needles, knitting needles, hooks
#     '7320': 'intermediate', # springs and leaves for springs
#     '7321': 'consumer product', # stoves, ranges, cooking appliances
#     '7322': 'consumer product', # radiators for central heating
#     '7323': 'consumer product', # table, kitchen, household articles of iron
#     '7324': 'consumer product', # sanitary ware of iron/steel
#     '7325': 'intermediate', # other cast articles
#     '7326': 'intermediate', # other articles of iron/steel
# }

# ironsteel_deep_dive = deep_dive_quad_chart(
#     ironsteel_results,
#     ironsteel_category_map,
#     group_name='Iron & Steel (HS 72/73)'
# )


=== Summary — Iron & Steel (HS 72/73) ===
        category  n_codes  total_air_wgt  avg_air_share  avg_value_per_kg  avg_mode_shift_opp  total_carbon_savings  total_adjusted_savings
consumer product        5        6735040       0.019833         46.794752            0.103565              0.032966                0.032966
    intermediate       43      262751037       0.019809         16.875802            0.407383              0.577402                0.286851
             raw        6        5321571       0.050535         13.425451            0.522477              0.014691                0.010046
           waste        1        3021984       0.000217          0.949551            0.826531              0.017871                0.017871
  Raw carbon savings:      0.6429 MtCO₂e
  Adjusted carbon savings: 0.3477 MtCO₂e


---
## 3. Electronic Machinery (HS Chapter 85)
Category map carried over from the original Florian analysis.
| Category | Example codes |
|---|---|
| intermediate | 8501–8507, 8511, 8514–8515, 8522, 8526, 8529–8538, 8540–8547 |
| consumer product | 8506, 8508–8510, 8512–8513, 8516–8521, 8523–8525, 8527–8528, 8531, 8539 |
| waste | 8548, 8549 |

In [17]:
elec_results = analyze_hs2_group(
    hs2_list=[85],
    all_hts_datasets=all_hts_datasets,
    group_name='Electronic Machinery'
)

  Loading distance cache from ocean_distances_cache.csv...
    Found 18944 cached routes
  Total unique routes: 10308
  Already cached: 10308
  Need to calculate: 0


/Users/aidangoldenberg-hart/Documents/GitHub/Maritime_Aviation_Repo/Scripts/functions.py:1165: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



Total hypothetical maritime emissions under mode shift: 0.03029440 MtCO2eq
Total aviation emissions for these shipments: 3.5749 MtCO2eq
Differential emissions (aviation - hypothetical maritime): 3.5446 MtCO2eq


In [45]:
elec_category_map = {
    '8501': 'intermediate',     # electric motors and generators
    '8502': 'intermediate',     # generating sets
    '8503': 'intermediate',     # parts for 8501/8502
    '8504': 'intermediate',     # electrical transformers, static converters
    '8505': 'intermediate',     # electromagnets; couplings, brakes
    '8506': 'consumer product', # primary cells and primary batteries
    '8507': 'intermediate',     # electric accumulators (rechargeable)
    '8508': 'consumer product', # vacuum cleaners
    '8509': 'consumer product', # electromechanical domestic appliances
    '8510': 'consumer product', # shavers, hair clippers, hair-removal devices
    '8511': 'intermediate',     # ignition/starting equipment for engines
    '8512': 'consumer product', # lighting/signalling equipment for vehicles
    '8513': 'consumer product', # portable electric lamps
    '8514': 'intermediate',     # industrial electric furnaces and ovens
    '8515': 'intermediate',     # industrial soldering/welding machines
    '8516': 'consumer product', # electric water heaters, hair dryers, irons
    '8517': 'consumer product', # telephone sets; smartphones; routers
    '8518': 'consumer product', # microphones, loudspeakers, amplifiers
    '8519': 'consumer product', # sound recording/reproducing apparatus
    '8520': 'consumer product', # magnetic tape recorders
    '8521': 'consumer product', # video recording/reproducing apparatus
    '8522': 'intermediate',     # parts/accessories for 8519–8521
    '8523': 'consumer product', # discs, tapes, smart cards for data storage
    '8524': 'consumer product', # flat panel display modules
    '8525': 'consumer product', # transmission apparatus; cameras
    '8526': 'intermediate',     # radar, radio navigation, remote control
    '8527': 'consumer product', # radio broadcast receivers
    '8528': 'consumer product', # monitors and projectors; TVs
    '8529': 'intermediate',     # parts for 8525–8528
    '8530': 'intermediate',     # electrical signalling equipment for railways
    '8531': 'consumer product', # electric sound/visual signalling apparatus
    '8532': 'intermediate',     # electrical capacitors
    '8533': 'intermediate',     # electrical resistors
    '8534': 'intermediate',     # printed circuits
    '8535': 'intermediate',     # electrical apparatus for switching (high V)
    '8536': 'intermediate',     # electrical apparatus for switching (<=1000V)
    '8537': 'intermediate',     # boards, panels, consoles for controlling
    '8538': 'intermediate',     # parts for 8535–8537
    '8539': 'consumer product', # electric filament or discharge lamps
    '8540': 'intermediate',     # thermionic valves and tubes
    '8541': 'intermediate',     # semiconductor devices; LEDs
    '8542': 'intermediate',     # electronic integrated circuits
    '8543': 'intermediate',     # other electrical machines with specific function
    '8544': 'intermediate',     # insulated wire, cable, optical fibre cables
    '8545': 'intermediate',     # carbon electrodes
    '8546': 'intermediate',     # electrical insulators
    '8547': 'intermediate',     # insulating fittings for electrical machines
    '8548': 'waste',            # waste/scrap of primary cells and batteries
    '8549': 'waste',            # electrical/electronic waste not elsewhere classified
}

elec_deep_dive = deep_dive_quad_chart(
    elec_results,
    elec_category_map,
    group_name='Electronic Machinery (HS 85)'
)


=== Summary — Electronic Machinery (HS 85) ===
        category  n_codes  total_air_wgt  avg_air_share  avg_value_per_kg  avg_mode_shift_opp  total_carbon_savings  total_adj_savings
consumer product       18      429346760       0.132180        158.006600            0.350026              0.832597           0.832597
    intermediate       28      702114491       0.204050        256.614701            0.294481              2.699481           1.207964
           waste        2        2833823       0.162357        100.339160            0.268412              0.012512           0.012512
  Raw carbon savings:      3.5446 MtCO₂e
  Adjusted carbon savings: 2.0531 MtCO₂e


---
## 4. Apparel — Knitted or Crocheted (HS Chapter 61)
All HS-4 codes in chapter 61 are finished consumer goods.
The category map still uses the four-label scheme for consistency;
add HS 62 codes if woven apparel is included in the dataset.

In [19]:
textiles_results = analyze_hs2_group(
    hs2_list=[61],
    all_hts_datasets=all_hts_datasets,
    group_name='Apparel (Knitted/Crocheted)'
)

  Loading distance cache from ocean_distances_cache.csv...
    Found 18944 cached routes
  Total unique routes: 4762
  Already cached: 4762
  Need to calculate: 0


/Users/aidangoldenberg-hart/Documents/GitHub/Maritime_Aviation_Repo/Scripts/functions.py:1165: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



Total hypothetical maritime emissions under mode shift: 0.00441864 MtCO2eq
Total aviation emissions for these shipments: 0.6443 MtCO2eq
Differential emissions (aviation - hypothetical maritime): 0.6399 MtCO2eq


In [46]:
apparel_category_map = {
    # ── HS 61: Knitted or Crocheted Clothing ──────────────────────────────
    # All codes are finished consumer products.
    '6101': 'consumer product', # men's overcoats, car coats, cloaks (knitted)
    '6102': 'consumer product', # women's overcoats, car coats, cloaks (knitted)
    '6103': 'consumer product', # men's suits, jackets, blazers, trousers (knitted)
    '6104': 'consumer product', # women's suits, jackets, dresses, skirts (knitted)
    '6105': 'consumer product', # men's shirts (knitted)
    '6106': 'consumer product', # women's blouses, shirts (knitted)
    '6107': 'consumer product', # men's underpants, nightwear, bathrobes (knitted)
    '6108': 'consumer product', # women's slips, nightwear, bathrobes (knitted)
    '6109': 'consumer product', # T-shirts, singlets and other vests (knitted)
    '6110': 'consumer product', # jerseys, pullovers, sweatshirts, cardigans (knitted)
    '6111': 'consumer product', # babies' garments and accessories (knitted)
    '6112': 'consumer product', # tracksuits, ski suits, swimwear (knitted)
    '6113': 'consumer product', # garments made of knitted/crocheted fabrics (nec)
    '6114': 'consumer product', # other garments (knitted)
    '6115': 'consumer product', # pantyhose, tights, stockings, socks (knitted)
    '6116': 'consumer product', # gloves, mittens and mitts (knitted)
    '6117': 'consumer product', # other made-up clothing accessories (knitted)
}

apparel_deep_dive = deep_dive_quad_chart(
    textiles_results,
    apparel_category_map,
    group_name='Apparel — Knitted/Crocheted (HS 61)'
)


=== Summary — Apparel — Knitted/Crocheted (HS 61) ===
        category  n_codes  total_air_wgt  avg_air_share  avg_value_per_kg  avg_mode_shift_opp  total_carbon_savings  total_adj_savings
consumer product       17      200130778       0.059482          29.92438            0.196528               0.63991            0.63991
  Raw carbon savings:      0.6399 MtCO₂e
  Adjusted carbon savings: 0.6399 MtCO₂e


---
## Cross-Group Comparison
Aggregate summary across all four commodity groups.

In [22]:
def group_summary_row(deep_dive_df, group_name):
    return {
        'group':                    group_name,
        'n_hs4_codes':              len(deep_dive_df),
        'total_air_weight_kg':      deep_dive_df['AIR_WGT_YR'].sum(),
        'mean_air_share':           deep_dive_df['air_share'].mean(),
        'mean_value_per_kg':        deep_dive_df['value_per_kg'].mean(),
        'mean_mode_shift_opp':      deep_dive_df['mode_shift_opportunity'].mean(),
        'total_carbon_savings_MtCO2': deep_dive_df['carbon_savings'].sum(),
        'adj_carbon_savings_MtCO2': deep_dive_df['adjusted_carbon_savings'].sum(),
    }

comparison = pd.DataFrame([
    group_summary_row(copper_deep_dive,    'Copper (74)'),
    group_summary_row(elec_deep_dive,      'Electronic Machinery (85)'),
    group_summary_row(apparel_deep_dive,   'Apparel (61)'),
])

comparison

,group,n_hs4_codes,total_air_weight_kg,mean_air_share,mean_value_per_kg,mean_mode_shift_opp,total_carbon_savings_MtCO2,adj_carbon_savings_MtCO2
0,Copper (74),16,25665634,0.061718,27.434238,0.456118,0.114092,0.070909
1,Electronic Machinery (85),48,1134295074,0.175361,213.125183,0.314224,3.544590,2.053073
2,Apparel (61),17,200130778,0.059482,29.924380,0.196528,0.639910,0.639910
